# Contrastive V1 — DISFA
Đọc `EXPERIMENT_PROTOCOL.md` và `COLAB_PRO.md` trước khi chạy. Notebook mặc định pilot 1 fold × 1 seed × 1 epoch, không đánh giá outer test. Chọn GPU trong Runtime. Upload code vào `MyDrive/SymAU/Contrastive_V1`, data archive vào `MyDrive/SymAU/DISFA.zip` (archive chứa thư mục DISFA).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, subprocess, sys, zipfile, os
PROJECT = Path('/content/Contrastive_V1')
DRIVE = Path('/content/drive/MyDrive/SymAU')
DATA = Path('/content/DISFA')
RUNS = DRIVE / 'runs'
SPLITS = DRIVE / 'splits'
if not PROJECT.exists():
    shutil.copytree(DRIVE / 'Contrastive_V1', PROJECT, ignore=shutil.ignore_patterns('.venv-tests', '__pycache__', '.pytest_cache', 'tmp'))
if not DATA.exists():
    with zipfile.ZipFile(DRIVE / 'DISFA.zip') as z:
        z.extractall('/content')  # your trusted archive
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'], check=True)
subprocess.run(['nvidia-smi'], check=True)
with (DRIVE / 'colab-environment.txt').open('w') as f:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=f, check=True)


In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU runtime first'
Path('checkpoints').mkdir(exist_ok=True)
weight = Path('checkpoints/resnet50-19c8e357.pth')
if not weight.exists():
    torch.hub.download_url_to_file('https://download.pytorch.org/models/resnet50-19c8e357.pth', str(weight))
subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q'], check=True)


## Lists và split
Nếu đã có lists đúng, giữ `FRAME_OFFSET=None`. Nếu chưa có lists, đọc hướng dẫn để xác nhận frame ID: annotation 1 tương ứng 0.png thì -1; 1.png thì 0. Tool không pad/skip frame và từ chối ghi đè lists khác.

In [ ]:
FRAME_OFFSET = None
if not (DATA / 'list' / 'DISFA_train_label_fold1.txt').exists():
    assert FRAME_OFFSET is not None, 'Confirm crop-to-annotation frame offset before generating lists'
    subprocess.run([sys.executable, 'prepare_disfa_lists.py', '--data-root', str(DATA), '--frame-offset', str(FRAME_OFFSET)], check=True)
subprocess.run([sys.executable, 'experiment_data.py', '--data-root', str(DATA), '--output', str(SPLITS), '--split-seed', '2026', '--val-subjects', '3'], check=True)


## Pilot → full study
Mặc định 3 pilot runs. Sau khi chốt batch/chi phí, đổi `STOP_AFTER=None`, `FOLDS=[1,2,3]`, `SEEDS=[0,1,2]`. Cell tự resume `last.pt`. Chia nhóm chạy theo ngân sách Colab, giữ tổng epochs=20. Thêm global/node/cross/global_cross nếu chạy ablation đầy đủ. Không chỉnh code/config giữa chừng một run.

In [ ]:
VARIANTS = ['baseline', 'current', 'full']
FOLDS = [1]
SEEDS = [0]
BATCH = 32
STOP_AFTER = 1
for variant in VARIANTS:
    for fold in FOLDS:
        for seed in SEEDS:
            out = RUNS / variant / f'fold{fold}' / f'seed{seed}'
            cmd = [sys.executable, 'train_contrastive.py', 'train', '--variant', variant, '--seed', str(seed), '--arc', 'resnet50', '--data-root', str(DATA), '--manifest', str(SPLITS/f'fold{fold}.json'), '--output', str(out), '--epochs', '20', '--batch-size', str(BATCH), '--workers', '2']
            if (out/'last.pt').exists():
                cmd += ['--resume', str(out/'last.pt')]
            if STOP_AFTER is not None:
                cmd += ['--stop-after', str(STOP_AFTER)]
            subprocess.run(cmd, check=True)


## Outer test — chỉ sau khi chốt thiết kế và train đủ ngân sách
Bật `RUN_OUTER_TEST=True` sau khi hoàn thành các runs. Runner kiểm tra đã train đủ planned epochs và checkpoint đúng best inner validation. Không dùng kết quả này để tune tiếp rồi báo trên chính test đó.

In [ ]:
RUN_OUTER_TEST = False
if RUN_OUTER_TEST:
    for variant in VARIANTS:
        for fold in FOLDS:
            for seed in SEEDS:
                out = RUNS / variant / f'fold{fold}' / f'seed{seed}'
                ev = out / 'evaluation'
                if not (ev/'test_metrics.json').exists():
                    subprocess.run([sys.executable, 'train_contrastive.py', 'evaluate', '--checkpoint', str(out/'best.pt'), '--data-root', str(DATA), '--manifest', str(SPLITS/f'fold{fold}.json'), '--output', str(ev), '--batch-size', str(BATCH)], check=True)


In [ ]:
if RUN_OUTER_TEST and set(FOLDS) == {1,2,3}:
    for baseline in ['baseline', 'current']:
        target = DRIVE / f'full_vs_{baseline}.json'
        if not target.exists():
            subprocess.run([sys.executable, 'summarize_experiments.py', '--root', str(RUNS), '--baseline', baseline, '--candidate', 'full', '--output', str(target)], check=True)
        print(target.read_text())


Các lệnh Stage 2–3 và giới hạn symbolic graph được ghi trong `COLAB_PRO.md`. Phải retrain GCN riêng cho từng Stage-1 checkpoint. Kết quả Stage 1 không đại diện tự động cho toàn pipeline.